#  Data Preparation - Rossmann Forecast Resilience

**Purpose:** This notebook prepares the raw Rossmann sales data for downstream forecasting experiments.

Before we can ask *'does an AI agent fix broken forecasts?'*, we need clean, forecastable data.  
This script turns raw daily pharmacy sales into **monthly store series** and selects only stores whose patterns a model can realistically learn.

---
**Run Order:** `data_preparation.ipynb` → `phase_a.ipynb` → `phase_b.ipynb` → `phase_c.ipynb`

| Input | Output |
|---|---|
| `train.csv` + `store.csv` (Kaggle) | `processed_monthly_data.csv` |
| | `mapse_denominators.pkl` |

## Cell 1 — Imports & Configuration

We import standard data science libraries and define the folder paths where raw data lives and where outputs should be saved.

>  **Update `DATA_FOLDER` and `OUTPUT_PATH`** to match your own machine before running.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import sys

# ── UPDATE THESE PATHS TO MATCH YOUR MACHINE ────────────────────
DATA_FOLDER = r"C:\Users\saisu\forecast-resilience\data"     # where train.csv and store.csv live
OUTPUT_PATH = r"C:\Users\saisu\forecast-resilience\outputs"  # where all outputs will be saved
# ────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(DATA_FOLDER, exist_ok=True)

print("✅ Libraries imported and paths configured.")
print(f"   Data folder : {DATA_FOLDER}")
print(f"   Output path : {OUTPUT_PATH}")

✅ Libraries imported and paths configured.
   Data folder : C:\Users\saisu\forecast-resilience\data
   Output path : C:\Users\saisu\forecast-resilience\outputs


## Cell 2 — Step 1: Load Raw Data

We load two files from the Kaggle Rossmann dataset:
- **`train.csv`** — daily sales for each store (the main time series)
- **`store.csv`** — store metadata (type, competition distance, etc.)

Both are needed together so that we can attach store characteristics to the sales history.

In [2]:
# Load the two raw CSV files and parse the Date column as proper datetime
try:
    train_df = pd.read_csv(
        os.path.join(DATA_FOLDER, "train.csv"),
        parse_dates=["Date"],
        dtype={"StateHoliday": str}   # keep holiday codes as strings (they mix letters and numbers)
    )
    store_df = pd.read_csv(os.path.join(DATA_FOLDER, "store.csv"))

    print(f"✅ Data loaded successfully.")
    print(f"   train.csv : {len(train_df):,} daily rows")
    print(f"   store.csv : {len(store_df)} stores")
    print(f"\n   Date range: {train_df['Date'].min().date()} → {train_df['Date'].max().date()}")
    train_df.head(3)

except FileNotFoundError as e:
    print(f"\n❌ ERROR: {e}")
    print(f"   Please put train.csv and store.csv in: {DATA_FOLDER}")
    sys.exit(1)

✅ Data loaded successfully.
   train.csv : 1,017,209 daily rows
   store.csv : 1115 stores

   Date range: 2013-01-01 → 2015-07-31


## Cell 3 — Step 2: Clean Data & Apply Forecastability Filter

### Why do we remove closed days?
When a store is closed, its sales are zero — but this is a *structural* zero (the store wasn't open), not a real demand signal. Including these zeros would make the series look more volatile than it actually is, which corrupts accuracy scores.

### What makes a store "forecastable"?
We apply four criteria to select only stores that a statistical model can realistically learn:

| Criterion | Threshold | Why |
|---|---|---|
| Trading days | ≥ 600 | Enough data to train and validate |
| Avg daily sales | ≥ €4,000 | Low-volume stores are too erratic |
| Coefficient of Variation | ≤ 50% | Not wildly volatile |
| Date span | ≥ 850 days | Covers the full experimental window |

From those that pass, we take the **top 20 by volume** — higher-volume stores have the most stable seasonal patterns.

In [3]:
# Remove days where the store was closed or recorded zero sales
train_df = train_df[(train_df["Open"] == 1) & (train_df["Sales"] > 0)].copy()
print(f"Rows after removing closed/zero days: {len(train_df):,}")

# Merge store metadata into the sales history
train_df = train_df.merge(store_df, on="Store", how="left")

# ── Compute per-store statistics ─────────────────────────────────
store_stats = train_df.groupby("Store").agg(
    total_days = ("Sales", "count"),
    avg_daily  = ("Sales", "mean"),
    std_daily  = ("Sales", "std"),
    min_date   = ("Date",  "min"),
    max_date   = ("Date",  "max"),
).reset_index()

# Coefficient of Variation: how volatile is the store relative to its own mean?
store_stats["cv_pct"]    = store_stats["std_daily"] / store_stats["avg_daily"] * 100
store_stats["date_span"] = (store_stats["max_date"] - store_stats["min_date"]).dt.days

# ── Apply the four forecastability criteria ───────────────────────
forecastable = store_stats[
    (store_stats["total_days"] >= 600) &
    (store_stats["avg_daily"]  >= 4000) &
    (store_stats["cv_pct"]     <= 50)  &
    (store_stats["date_span"]  >= 850)
]["Store"].tolist()

print(f"Stores passing forecastability filter: {len(forecastable)}")

# Take the top 20 by average daily volume
top20 = (
    store_stats[store_stats["Store"].isin(forecastable)]
    .nlargest(20, "avg_daily")["Store"].tolist()
)

train_df = train_df[train_df["Store"].isin(top20)].copy()
print(f"✅ Using top 20 stores by average daily volume.")

Rows after removing closed/zero days: 844,338
Stores passing forecastability filter: 1049
✅ Using top 20 stores by average daily volume.


## Cell 4 - Step 3: Aggregate to Monthly Revenue

### Why aggregate to monthly?
Daily pharmacy sales are dominated by day-of-week effects (weekends are different from weekdays) and short-term promotional noise. For supply-chain planning, the **monthly horizon** is what matters — monthly data removes that short-term noise so that seasonal and trend patterns dominate, making the model's job much cleaner.

### Normalising to a 30-day equivalent
Not every month has the same number of trading days (e.g. February vs. December). We normalise each month's total to a *'what would this store earn in exactly 30 trading days?'* equivalent, so months are fairly comparable.

In [4]:
# Create a year-month period column for grouping
train_df["year_month"] = train_df["Date"].dt.to_period("M")

# Aggregate daily rows into monthly summaries
monthly = train_df.groupby(["Store", "year_month"]).agg(
    sales         = ("Sales",        "sum"),    # total monthly revenue
    holiday_count = ("SchoolHoliday", "sum"),   # number of school-holiday days that month
    promo_days    = ("Promo",         "sum"),   # number of promotional days
    trading_days  = ("Open",          "count"), # actual open days (denominator for normalisation)
).reset_index()

monthly["date"] = monthly["year_month"].dt.to_timestamp()

# Normalise to 30-day equivalent so shorter/longer months are comparable
monthly["sales"] = (monthly["sales"] / monthly["trading_days"] * 30).round(0)

# ── Restrict to Jan 2013 → Jul 2015 (the experimental window) ────
START = pd.Timestamp("2013-01-01")
END   = pd.Timestamp("2015-07-01")
monthly = monthly[(monthly["date"] >= START) & (monthly["date"] <= END)].copy()

# Drop any store that has a missing month — gaps break ARIMA silently
month_counts = monthly.groupby("Store")["date"].count()
n_expected   = len(pd.date_range(START, END, freq="MS"))
complete     = month_counts[month_counts == n_expected].index.tolist()
monthly      = monthly[monthly["Store"].isin(complete)].copy()

print(f"Expected months per store : {n_expected}")
print(f"Stores with complete data : {len(complete)}")

# Attach the series IDs used by Phase A/B/C
monthly["id"] = monthly["Store"].apply(lambda s: f"ROSSMANN_STORE_{s:04d}_validation")

# Build the final output dataframe
out_df = monthly[["id", "date", "sales", "holiday_count"]].copy()
out_df = out_df.sort_values(["id", "date"]).reset_index(drop=True)

print(f"\n✅ Monthly data prepared.")
print(f"   Rows   : {len(out_df):,}  |  Stores: {out_df['id'].nunique()}")
print(f"   Range  : {out_df['date'].min().date()} → {out_df['date'].max().date()}")
out_df.head()

Expected months per store : 31
Stores with complete data : 19

✅ Monthly data prepared.
   Rows   : 589  |  Stores: 19
   Range  : 2013-01-01 → 2015-07-01


,id,date,sales,holiday_count
0,ROSSMANN_STORE_0251_validation,2013-01-01,546043.0,3
1,ROSSMANN_STORE_0251_validation,2013-02-01,567921.0,0
2,ROSSMANN_STORE_0251_validation,2013-03-01,620940.0,4
3,ROSSMANN_STORE_0251_validation,2013-04-01,557236.0,4
4,ROSSMANN_STORE_0251_validation,2013-05-01,616717.0,1


## Cell 5 — Step 4: Pre-compute MAPSE Denominators & Save Outputs

### What is MAPSE and why do we pre-compute its denominator?
**MAPSE (Mean Absolute Percentage Scaled Error)** normalises each store's forecast error by that store's own average sales during the *training period*.

This lets us compare stores fairly:
- A €650k/month store and a €420k/month store both land on the **same 0–100% scale**
- Without this normalisation, larger stores would dominate the average error

We compute the denominator **once here** and freeze it — so it stays constant across Phase A, B, and C experiments, ensuring all comparisons are apples-to-apples.

$$\text{MAPSE} = \frac{|\text{actual} - \text{forecast}|}{\text{training mean}} \times 100\%$$

In [5]:
# Save processed monthly data to CSV
out_csv = os.path.join(OUTPUT_PATH, "processed_monthly_data.csv")
out_df.to_csv(out_csv, index=False)
print(f"✅ Saved: processed_monthly_data.csv")

# ── Pre-compute MAPSE denominators ───────────────────────────────
TEST_MONTHS = 6                                        # last 6 months are held out as the test set
all_dates   = sorted(out_df["date"].unique())
split_date  = pd.Timestamp(all_dates[-TEST_MONTHS])   # the train/test boundary

print(f"\n   Test window  : last {TEST_MONTHS} months ({split_date.date()} onward)")
print(f"   Train window : up to {(split_date - pd.offsets.MonthBegin(1)).date()}")

# For each store, compute the mean of its training-period sales
mapse_denoms = {}
for sid in out_df["id"].unique():
    train_vals = out_df[
        (out_df["id"] == sid) & (out_df["date"] < split_date)
    ]["sales"].values
    mapse_denoms[sid] = float(np.mean(train_vals)) if len(train_vals) > 0 else 1.0

# Save denominators as a pickle file so Phase A/B/C can load them instantly
denom_path = os.path.join(OUTPUT_PATH, "mapse_denominators.pkl")
with open(denom_path, "wb") as f:
    pickle.dump(mapse_denoms, f)
print(f"✅ Saved: mapse_denominators.pkl ({len(mapse_denoms)} stores)")

✅ Saved: processed_monthly_data.csv

   Test window  : last 6 months (2015-02-01 onward)
   Train window : up to 2015-01-01
✅ Saved: mapse_denominators.pkl (19 stores)


## Cell 6 - Store Health Preview

Before handing the data to Phase A, we do a quick sanity check on every store.

**Expected MAPSE rule of thumb:** a well-fitted monthly model typically achieves around **40% of the series' Coefficient of Variation (CV)**. So if a store has CV = 25%, we'd expect MAPSE ≈ 10% → GREEN.

| Band | MAPSE | Meaning |
|---|---|---|
| 🟢 GREEN | < 20% | Model should forecast well |
| 🟡 YELLOW | 20–35% | Acceptable, some noise |
| 🔴 RED | ≥ 35% | Too erratic — should not be included |

In [6]:
print(f"{'Store ID':<38} {'Avg Sales':>12}  {'CV%':>5}  Expected MAPSE")
print(f"{'-'*70}")

all_ok = True
for sid in sorted(out_df["id"].unique()):
    ser     = out_df[out_df["id"] == sid]["sales"]
    avg     = ser.mean()
    cv      = ser.std() / ser.mean() * 100
    exp_m   = cv * 0.4   # rule of thumb: a good model hits ~40% of CV

    # Traffic light based on expected MAPSE
    colour  = f"🟢 GREEN  ~{exp_m:.0f}%"  if exp_m < 20 else \
              f"🟡 YELLOW ~{exp_m:.0f}%" if exp_m < 35 else \
              f"🔴 RED    ~{exp_m:.0f}%"
    if exp_m >= 35:
        all_ok = False
    print(f"  {sid:<38} {avg:>12,.0f}  {cv:>5.1f}%  {colour}")

print(f"\n{'✅ All stores look forecastable — ready for Phase A.' if all_ok else '⚠️ Some stores may be noisy — Phase A will confirm.'}")
print(f"\n➡️  NEXT STEP: Open phase_a.ipynb")

Store ID                                  Avg Sales    CV%  Expected MAPSE
----------------------------------------------------------------------
  ROSSMANN_STORE_0251_validation              574,866    8.9%  🟢 GREEN  ~4%
  ROSSMANN_STORE_0262_validation              621,278    7.5%  🟢 GREEN  ~3%
  ROSSMANN_STORE_0320_validation              445,520   10.0%  🟢 GREEN  ~4%
  ROSSMANN_STORE_0336_validation              417,878    6.9%  🟢 GREEN  ~3%
  ROSSMANN_STORE_0380_validation              434,505    8.3%  🟢 GREEN  ~3%
  ROSSMANN_STORE_0383_validation              519,445    5.6%  🟢 GREEN  ~2%
  ROSSMANN_STORE_0513_validation              545,945    7.1%  🟢 GREEN  ~3%
  ROSSMANN_STORE_0523_validation              466,276   15.3%  🟢 GREEN  ~6%
  ROSSMANN_STORE_0544_validation              423,240    8.1%  🟢 GREEN  ~3%
  ROSSMANN_STORE_0562_validation              539,092    5.7%  🟢 GREEN  ~2%
  ROSSMANN_STORE_0586_validation              454,663   12.3%  🟢 GREEN  ~5%
  ROSSMANN_STORE_0

Results Interpretation: AI-Driven Forecast Resilience Framework
Data Preparation
The dataset was carefully filtered from over one million daily records down to 19 pharmacies that met strict trading consistency thresholds. Monthly aggregation was chosen deliberately to expose structural demand patterns rather than short-term noise. This matters for the project because it means any degradation observed later in Phase B is genuinely caused by disruptions, not by messy underlying data. The MAPSE metric scales errors store-by-store, which makes comparisons fair across stores of different revenue sizes.
Why this matters for resilience:

A clean, well-filtered baseline is the foundation that makes stress testing credible
Monthly aggregation removes day-level noise so the resilience tests isolate exactly what they are supposed to isolate
Store-level MAPSE denominators prevent larger stores from masking problems in smaller ones